### Qurry enhancement -> query expasion

In [2]:
from langchain_community.document_loaders import  TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models.base import init_chat_model
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

d:\RAG_Learning_Repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader=TextLoader("langchain_crewai_dataset.txt")
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
chunks=text_splitter.split_documents(documents)
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embeddings)
retriever=vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 618.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
llm=init_chat_model("groq:openai/gpt-oss-safeguard-20b")
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000244934D4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000244934D63C0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant keywords and context.

Original query: "{query}"

Expanded query:
""")
query_expansion_chain=query_expansion_prompt | llm| StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant keywords and context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000244934D4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000244934D63C0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [6]:
response = query_expansion_chain.invoke({"query":"langchain memory"})
print(response)


**Expanded query:**

“LangChain memory usage and implementation details – including ConversationBufferMemory, RetrievalQA memory, memory store options, vector‑store integration, memory management best practices, examples, performance tuning, and how memory is used in RAG and chatbot pipelines.”


In [8]:
from langchain_core.runnables import RunnableMap
prompt_answer=PromptTemplate.from_template("""
answer the question based on the following retrieved contexts:
contexts:{context}
question:{input}                                       
                                                                               
""")
chain_documents=create_stuff_documents_chain(llm,prompt_answer)
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | chain_documents
)

response=rag_pipeline.invoke({"input":"how does langchain memory work?"})
print(response)

**LangChain’s memory system lets an LLM “remember” what’s happened in a conversation or chain of steps, so it can make context‑aware decisions and keep track of long interactions.**  

| Memory type | What it stores | How it’s used | Typical use‑case |
|--------------|----------------|---------------|------------------|
| **ConversationBufferMemory** | The raw, unaltered text of every turn in the conversation | The buffer is appended to each prompt sent to the LLM, giving it the full history. | When the token budget allows it and you need precise recall of everything said. |
| **ConversationSummaryMemory** | A compressed, summarized version of the conversation | The summary replaces or augments the raw history in the prompt, keeping the most important context while staying within token limits. | Long‑running sessions where the raw dialogue would exceed token limits, or when you want the model to focus on key points rather than every detail. |

### How it works in practice

1. **Initial